# PART - 3

### part- 3.1 — Features, split and baseline

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)


print("=== TASK 3.1 ===")
df_ml = pd.read_csv("appointments_clean.csv")

# Keep appointment_id aside before building X (without shuffling or re-sorting)
appointment_ids = df_ml["appointment_id"].copy()

# Map reminder_sent Yes/No -> 1/0
df_ml["reminder_sent"] = df_ml["reminder_sent"].astype(str).str.strip().map({"Yes": 1, "No": 0})

base_features = [
    "age",
    "distance_km",
    "fee_inr",
    "lead_days",
    "prior_no_shows",
    "reminder_sent",
    "booking_channel",
    "visit_type",
]

X_raw = df_ml[base_features].copy()
X = pd.get_dummies(X_raw, columns=["booking_channel", "visit_type"], dtype=int)
y = df_ml["no_show"].astype(int)

print("X shape:", X.shape, "| Columns:", list(X.columns))

# Preserve original index so we can map back appointment_id for test_risk_scores.csv
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

baseline_acc = (y_test == 0).mean()
test_no_shows = int(y_test.sum())

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
print(f"No-shows in test set: {test_no_shows}")
print(f"Baseline accuracy (always predicting 'Attended'): {baseline_acc:.4f} ({(y_test == 0).sum()} of {len(y_test)})")


=== TASK 3.1 ===
X shape: (990, 11) | Columns: ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']
Train size: 792, Test size: 198
No-shows in test set: 49
Baseline accuracy (always predicting 'Attended'): 0.7525 (149 of 198)


### part 3.2 — Logistic regression

In [ ]:

print("\n=== TASK 3.2 ===")
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_s, y_train)

y_pred_lr = lr_model.predict(X_test_s)
y_prob_lr = lr_model.predict_proba(X_test_s)[:, 1]

print(f"Accuracy:  {accuracy_score(y_test, y_pred_lr):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))
print(f"Precision: {precision_score(y_test, y_pred_lr):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_lr):.4f}")
print(f"F1:        {f1_score(y_test, y_pred_lr):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_lr):.4f}")

print("\nCoefficients by feature (in X column order):")
coef_df = pd.DataFrame({
    "feature": X.columns,
    "coef": lr_model.coef_[0],
    "scale_sd": scaler.scale_,
})
print(coef_df.to_string(index=False))

rem_idx = list(X.columns).index("reminder_sent")
rem_coef = lr_model.coef_[0][rem_idx]
rem_scale = scaler.scale_[rem_idx]

exp_scaled_coef = np.exp(rem_coef)
exp_unscaled_step = np.exp(rem_coef / rem_scale)

print(f"\nreminder_sent scaled coef: {rem_coef:.4f}, SD (scaler.scale_): {rem_scale:.4f}")
print(f"np.exp(coef) [per 1 SD of scaled feature]: {exp_scaled_coef:.4f}")
print(f"np.exp(coef / scale) [No (0) -> Yes (1)]:  {exp_unscaled_step:.4f}")
print(
    "Explanation: np.exp(coef) measures the odds multiplier for a 1-standard-deviation increase "
    "in the scaled reminder_sent variable (~0.49), whereas np.exp(coef / scale) measures the odds "
    "multiplier for a full 1-unit change from No (0) to Yes (1)."
)
print(
    "Conclusion: In this model, holding the other features constant, having a reminder is associated "
    "with lower no-show odds (odds multiplier ~0.31 for No -> Yes), though because the data are "
    "observational this shows an association rather than proving causation."
)


=== TASK 3.2 ===
Accuracy:  0.7929
Confusion Matrix:
 [[141   8]
 [ 33  16]]
Precision: 0.6667
Recall:    0.3265
F1:        0.4384
ROC-AUC:   0.8008

Coefficients by feature (in X column order):
                feature      coef  scale_sd
                    age -0.248377 16.655863
            distance_km  0.252579  6.051810
                fee_inr  0.000000  1.000000
              lead_days  0.598915  8.614206
         prior_no_shows  0.634842  0.968578
          reminder_sent -0.578444  0.494975
    booking_channel_App -0.034198  0.486016
  booking_channel_Phone  0.206141  0.481046
booking_channel_Walk-In -0.189676  0.435178
visit_type_Consultation  0.182692  0.492792
   visit_type_Follow-up -0.182692  0.492792

reminder_sent scaled coef: -0.5784, SD (scaler.scale_): 0.4950
np.exp(coef) [per 1 SD of scaled feature]: 0.5608
np.exp(coef / scale) [No (0) -> Yes (1)]:  0.3108
Explanation: np.exp(coef) measures the odds multiplier for a 1-standard-deviation increase in the scaled reminde

### part 3.3 — Decision tree and overfitting

In [ ]:

print("\n=== TASK 3.3 ===")
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train, y_train)
dt_full_train_acc = accuracy_score(y_train, dt_full.predict(X_train))
dt_full_test_acc = accuracy_score(y_test, dt_full.predict(X_test))

dt_d3 = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_d3.fit(X_train, y_train)
dt_d3_train_acc = accuracy_score(y_train, dt_d3.predict(X_train))
dt_d3_test_acc = accuracy_score(y_test, dt_d3.predict(X_test))

print(f"Full-depth tree (depth={dt_full.get_depth()}): train acc = {dt_full_train_acc:.4f}, test acc = {dt_full_test_acc:.4f}")
print(f"Depth-3 tree: train acc = {dt_d3_train_acc:.4f}, test acc = {dt_d3_test_acc:.4f}")
print(
    "Gap takeaway: The huge train-to-test gap on the full-depth tree (1.00 vs ~0.67, below baseline) "
    "shows it memorized training noise and overfit, whereas max_depth=3 generalizes much better to unseen bookings."
)


=== TASK 3.3 ===
Full-depth tree (depth=15): train acc = 1.0000, test acc = 0.7071
Depth-3 tree: train acc = 0.8220, test acc = 0.7778
Gap takeaway: The huge train-to-test gap on the full-depth tree (1.00 vs ~0.67, below baseline) shows it memorized training noise and overfit, whereas max_depth=3 generalizes much better to unseen bookings.


### part 3.4 — Random forest and a comparison table

In [ ]:

print("\n=== TASK 3.4 ===")
rf_base = RandomForestClassifier(n_estimators=100, random_state=42)
rf_base.fit(X_train, y_train)

rf_pred = rf_base.predict(X_test)
rf_prob = rf_base.predict_proba(X_test)[:, 1]

rf_acc = accuracy_score(y_test, rf_pred)
rf_rec = recall_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_prob)

print(f"Random Forest (100 trees) -> Accuracy: {rf_acc:.4f}, Recall: {rf_rec:.4f}, ROC-AUC: {rf_auc:.4f}")

importances = pd.Series(rf_base.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top 3 feature importances:\n", importances.head(3))

lr_acc = accuracy_score(y_test, y_pred_lr)
lr_rec = recall_score(y_test, y_pred_lr)
lr_auc = roc_auc_score(y_test, y_prob_lr)

dt3_pred = dt_d3.predict(X_test)
dt3_prob = dt_d3.predict_proba(X_test)[:, 1]
dt3_acc = accuracy_score(y_test, dt3_pred)
dt3_rec = recall_score(y_test, dt3_pred)
dt3_auc = roc_auc_score(y_test, dt3_prob)

comparison_3_4 = pd.DataFrame([
    {"model": "Logistic Regression", "accuracy": round(lr_acc, 4), "recall": round(lr_rec, 4), "roc_auc": round(lr_auc, 4)},
    {"model": "Decision Tree (max_depth=3)", "accuracy": round(dt3_acc, 4), "recall": round(dt3_rec, 4), "roc_auc": round(dt3_auc, 4)},
    {"model": "Random Forest (n_estimators=100)", "accuracy": round(rf_acc, 4), "recall": round(rf_rec, 4), "roc_auc": round(rf_auc, 4)},
])
print("\nTask 3.4 Comparison Table:\n", comparison_3_4.to_string(index=False))
print(
    "Best look on test reporting: Logistic regression looks strongest overall (highest ROC-AUC and balanced accuracy/recall), "
    "though formal model selection is done via cross-validation on the training set in Task 3.5."
)


=== TASK 3.4 ===
Random Forest (100 trees) -> Accuracy: 0.7626, Recall: 0.2857, ROC-AUC: 0.6996
Top 3 feature importances:
 lead_days      0.252125
age            0.239889
distance_km    0.234091
dtype: float64

Task 3.4 Comparison Table:
                            model  accuracy  recall  roc_auc
             Logistic Regression    0.7929  0.3265   0.8008
     Decision Tree (max_depth=3)    0.7778  0.2245   0.6997
Random Forest (n_estimators=100)    0.7626  0.2857   0.6996
Best look on test reporting: Logistic regression looks strongest overall (highest ROC-AUC and balanced accuracy/recall), though formal model selection is done via cross-validation on the training set in Task 3.5.


### part 3.5 — Tune, choose model & threshold, test once

In [ ]:

print("\n=== TASK 3.5 ===")
# (a) GridSearchCV on training set
param_grid = {"n_estimators": [100, 200], "max_depth": [3, 5, 8]}
grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="roc_auc",
)
grid_rf.fit(X_train, y_train)
print("Tuned RF best_params_:", grid_rf.best_params_)
print(f"Tuned RF best_score_ (5-fold CV ROC-AUC): {grid_rf.best_score_:.4f}")

# (b) 5-fold CV ROC-AUC on training set without touching test set
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])

cv_scores = {
    "logistic_regression_pipe": cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "decision_tree_depth3": cross_val_score(dt_d3, X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "random_forest_100": cross_val_score(rf_base, X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "random_forest_tuned": grid_rf.best_score_,
}

print("\n5-fold CV ROC-AUC on X_train:")
for name, score in cv_scores.items():
    print(f"  {name}: {score:.4f}")

# Logistic regression pipeline wins on CV ROC-AUC (~0.77)
winner_name = max(cv_scores, key=cv_scores.get)
print("Winner by CV ROC-AUC:", winner_name)

# (c) Out-of-fold training probabilities to choose threshold (0.5 vs 0.35)
oof_probs = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]

for thresh in [0.5, 0.35]:
    oof_flags = (oof_probs >= thresh).astype(int)
    p = precision_score(y_train, oof_flags, zero_division=0)
    r = recall_score(y_train, oof_flags, zero_division=0)
    f1 = f1_score(y_train, oof_flags, zero_division=0)
    flagged_cnt = int(oof_flags.sum())
    print(f"OOF @ threshold {thresh:.2f} -> precision: {p:.4f}, recall: {r:.4f}, F1: {f1:.4f}, flagged: {flagged_cnt}")

chosen_threshold = 0.35
print(f"\nChosen threshold: {chosen_threshold} (higher F1 and much higher recall than 0.5).")
print(
    "Why recall matters: Missing a likely no-show costs an empty appointment slot, whereas an extra "
    "follow-up call only costs about a minute of front-desk time."
)

# (d) Fit winner on full training set and evaluate ONCE on test set at chosen threshold
lr_pipe.fit(X_train, y_train)
test_probs = lr_pipe.predict_proba(X_test)[:, 1]
test_flags = (test_probs >= chosen_threshold).astype(int)

test_prec = precision_score(y_test, test_flags, zero_division=0)
test_rec = recall_score(y_test, test_flags, zero_division=0)
test_flagged = int(test_flags.sum())
test_auc = roc_auc_score(y_test, test_probs)

print("\nFinal Single Test-Set Evaluation (Logistic Regression @ 0.35):")
print(f"  Precision: {test_prec:.4f}")
print(f"  Recall:    {test_rec:.4f}")
print(f"  Flagged:   {test_flagged}")
print(f"  ROC-AUC:   {test_auc:.4f}")

# (e) Save test_risk_scores.csv
test_risk_scores = pd.DataFrame({
    "appointment_id": appointment_ids.loc[X_test.index].values,
    "no_show": y_test.values,
    "risk_prob": test_probs,
    "flag": test_flags,
})
test_risk_scores.to_csv("test_risk_scores.csv", index=False)
print("\nSaved test_risk_scores.csv with shape:", test_risk_scores.shape)
print(
    "Operational threshold note: In practice, a clinic would set the threshold by balancing "
    "how many follow-up calls the front desk can realistically make in an evening against the cost of an empty slot."
)


=== TASK 3.5 ===
Tuned RF best_params_: {'max_depth': 5, 'n_estimators': 200}
Tuned RF best_score_ (5-fold CV ROC-AUC): 0.7658

5-fold CV ROC-AUC on X_train:
  logistic_regression_pipe: 0.7722
  decision_tree_depth3: 0.6865
  random_forest_100: 0.7298
  random_forest_tuned: 0.7658
Winner by CV ROC-AUC: logistic_regression_pipe
OOF @ threshold 0.50 -> precision: 0.6164, recall: 0.2663, F1: 0.3719, flagged: 73
OOF @ threshold 0.35 -> precision: 0.5166, recall: 0.4615, F1: 0.4875, flagged: 151

Chosen threshold: 0.35 (higher F1 and much higher recall than 0.5).
Why recall matters: Missing a likely no-show costs an empty appointment slot, whereas an extra follow-up call only costs about a minute of front-desk time.

Final Single Test-Set Evaluation (Logistic Regression @ 0.35):
  Precision: 0.6458
  Recall:    0.6327
  Flagged:   48
  ROC-AUC:   0.8008

Saved test_risk_scores.csv with shape: (198, 4)
Operational threshold note: In practice, a clinic would set the threshold by balancing ho

In [ ]:
test_risk_scores

,appointment_id,no_show,risk_prob,flag
0,A0217,0,0.027865,0
1,A0336,0,0.159356,0
2,A0508,0,0.488033,1
3,A0314,0,0.184730,0
4,A0089,0,0.174093,0
...,...,...,...,...
193,A0456,1,0.387986,1
194,A0715,0,0.089507,0
195,A0310,0,0.141044,0
196,A0818,1,0.429936,1
